# 雙板 RFSoC 4×2 OFDM 圖片收發

兩塊板各解壓一份發布包。用兩條 50 Ω SMA 同軸線及 10 dB 衰減器交叉接線。

| 路徑 | 接線 |
| --- | --- |
| 圖片資料 | A `DAC_A` → 衰減器 → B `ADC_A` |
| RF ACK | B `DAC_A` → 衰減器 → A `ADC_A` |

在兩塊板的 JupyterLab 各開一個 terminal，進入解壓資料夾。完整接線與故障排查請見發布包的 [README.md](README.md)。

## 使用兩個 terminal 傳圖

1. 在板 B 執行 `sudo -E python3 run_on_board.py receive`，等候 `RX_WAITING_FOR_PILOT`。
2. 在板 A 執行 `sudo -E python3 run_on_board.py send`。兩塊板會自動交換 pilot。
3. 板 B 顯示 `RX_ARMED` 後，在板 A 按 Enter 傳圖。
4. 板 A 顯示 `TX_DONE` 後，在板 B 按 Enter。板 B 此時停止接收、匯出圖片與封包品質並顯示 `RX_DONE`。

每包最多嘗試 11 次。耗盡後 A 會繼續下一包。B 會輸出固定尺寸的圖片，使用最近收到且標頭驗證正確的完整錯包作為備用資料，完全缺少的包填 0。

可在板 A 的命令加上 `--input photo.jpg` 更換圖片。

## 查看板 B 的接收圖

在板 B 開啟這份 Notebook，等傳輸完成後再執行下一格。若使用了 `--output`，請修改結果目錄。

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

rx_dir = Path("rx_results")
rx = json.loads((rx_dir / "result.json").read_text(encoding="utf-8"))
if rx.get("role") != "receive" or not rx.get("completed"):
    raise RuntimeError(f"Check {rx_dir / 'bringup.log'} and wait for RX_DONE on board B.")
display(rx)
display(Image(filename=str(rx_dir / "received.png")))

## 比對兩板實際的像素

將 A 的 `tx_results/`、B 的 `rx_results/` 和 `run_on_board.py` 放在同一台電腦的資料夾。從該資料夾執行以下指令。

```bash
python run_on_board.py verify --tx-dir tx_results --rx-dir rx_results --output two_board_result.json
```

完整無誤時 `passed` 為 `true`。報告也會列出 `mismatched_bytes`、`crc_bad_packet_ids` 與 `missing_packet_ids`，有錯包或缺包時仍可查看已匯出的圖片。